In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from scipy.sparse import csr_matrix, hstack

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


# ============================================================
# CONFIGURATION
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

MODEL_INPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
)

LINKED_ROOT = (
    PROJECT_ROOT
    / "data"
    / "linked"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "price_prediction"
    / "external_attributes_v4"
)

REPORT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "reports"
    / "price_prediction"
    / "external_attributes_v4"
)

MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RANDOM_STATE = 42

CURRENT_BEST_VALIDATION_MAE = 21.8504


# ============================================================
# FIND LINKED DATASET
# ============================================================

candidate_files = [
    LINKED_ROOT / "multimodal_products_direct.parquet",
    LINKED_ROOT / "linked_products_direct.parquet",
]

LINKED_FILE = None

for candidate in candidate_files:
    if candidate.exists():
        LINKED_FILE = candidate
        break

if LINKED_FILE is None:
    raise FileNotFoundError(
        "Could not find multimodal_products_direct.parquet "
        "or linked_products_direct.parquet."
    )

print("=" * 90)
print("V4 EXTERNAL ATTRIBUTE ENRICHMENT")
print("=" * 90)

print("Linked dataset:")
print(LINKED_FILE)


# ============================================================
# LOAD MODEL SPLITS
# ============================================================

train_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "train.parquet"
).reset_index(drop=True)

validation_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "validation.parquet"
).reset_index(drop=True)

test_df = pd.read_parquet(
    MODEL_INPUT_ROOT / "test.parquet"
).reset_index(drop=True)

print("\nModel data:")
print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)


# ============================================================
# INSPECT LINKED DATASET SCHEMA
# ============================================================

schema_columns = (
    pq.ParquetFile(
        LINKED_FILE
    )
    .schema
    .names
)

print("\nLinked columns available:")

for column in schema_columns:
    print(" ", column)


# ============================================================
# EXTERNAL COLUMNS WE WANT
# ============================================================

desired_external_columns = [
    "asin",
    "brand",
    "color",
    "material",
    "mave_attribute_count",
    "mave_attributes_json",
]

available_external_columns = [
    column
    for column in desired_external_columns
    if column in schema_columns
]

if "asin" not in available_external_columns:
    raise ValueError(
        "Linked dataset does not contain 'asin'."
    )

print("\nExternal columns selected:")
print(available_external_columns)


# ============================================================
# LOAD ONLY REQUIRED LINKED COLUMNS
# ============================================================

external_df = pd.read_parquet(
    LINKED_FILE,
    columns=available_external_columns,
)

external_df["asin"] = (
    external_df["asin"]
    .fillna("")
    .astype(str)
)

external_df = (
    external_df
    .drop_duplicates(
        subset=["asin"],
        keep="first",
    )
)

print(
    "\nExternal rows:",
    f"{len(external_df):,}"
)


# ============================================================
# ENSURE ALL OPTIONAL COLUMNS EXIST
# ============================================================

for column in [
    "brand",
    "color",
    "material",
    "mave_attribute_count",
    "mave_attributes_json",
]:

    if column not in external_df.columns:
        external_df[column] = np.nan


# ============================================================
# MERGE WITH EACH SPLIT
# ============================================================

def merge_external_attributes(df):

    df = df.copy()

    df["asin"] = (
        df["asin"]
        .fillna("")
        .astype(str)
    )

    return df.merge(
        external_df,
        on="asin",
        how="left",
        validate="one_to_one",
    )


train_df = merge_external_attributes(
    train_df
)

validation_df = merge_external_attributes(
    validation_df
)

test_df = merge_external_attributes(
    test_df
)


# ============================================================
# COVERAGE ANALYSIS
# ============================================================

def coverage_report(
    df,
    split_name,
):

    rows = []

    for column in [
        "brand",
        "color",
        "material",
        "mave_attributes_json",
    ]:

        valid = (
            df[column]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
        )

        rows.append(
            {
                "split": split_name,
                "field": column,
                "available_rows": int(
                    valid.sum()
                ),
                "total_rows": len(df),
                "coverage_percent": (
                    valid.mean() * 100
                ),
            }
        )

    return rows


coverage_rows = []

coverage_rows.extend(
    coverage_report(
        train_df,
        "train",
    )
)

coverage_rows.extend(
    coverage_report(
        validation_df,
        "validation",
    )
)

coverage_rows.extend(
    coverage_report(
        test_df,
        "test",
    )
)

coverage_df = pd.DataFrame(
    coverage_rows
)

print()
print("=" * 90)
print("ABO / MAVE COVERAGE")
print("=" * 90)

display(
    coverage_df
)

coverage_df.to_csv(
    REPORT_ROOT
    / "external_attribute_coverage.csv",
    index=False,
)


# ============================================================
# FLATTEN MAVE JSON INTO TEXT
# ============================================================

def flatten_mave_attributes(value):

    if pd.isna(value):
        return ""

    value = str(value).strip()

    if not value:
        return ""

    try:
        parsed = json.loads(value)

    except Exception:
        return value

    parts = []

    if isinstance(parsed, dict):

        for key, values in parsed.items():

            if isinstance(
                values,
                list,
            ):

                clean_values = [
                    str(v).strip()
                    for v in values
                    if str(v).strip()
                ]

                if clean_values:

                    parts.append(
                        f"{key} "
                        + " ".join(
                            clean_values
                        )
                    )

            elif values is not None:

                parts.append(
                    f"{key} {values}"
                )

    return " ".join(parts)


# ============================================================
# BASE + EXTERNAL FEATURE ENGINEERING
# ============================================================

def create_features(df):

    df = df.copy()

    # --------------------------------------------------------
    # CLEAN BASIC FIELDS
    # --------------------------------------------------------

    df["title"] = (
        df["title"]
        .fillna("")
        .astype(str)
    )

    df["category_name"] = (
        df["category_name"]
        .fillna("Unknown")
        .astype(str)
    )

    df["stars"] = pd.to_numeric(
        df["stars"],
        errors="coerce",
    ).fillna(0)

    reviews = pd.to_numeric(
        df["reviews"],
        errors="coerce",
    ).fillna(0)

    bought = pd.to_numeric(
        df["boughtInLastMonth"],
        errors="coerce",
    ).fillna(0)

    df["reviews_log1p"] = np.log1p(
        reviews
    )

    df["bought_log1p"] = np.log1p(
        bought
    )

    df["isBestSeller"] = (
        df["isBestSeller"]
        .fillna(False)
        .astype(int)
    )

    df["cluster_id"] = pd.to_numeric(
        df["cluster_id"],
        errors="coerce",
    ).fillna(-1)

    # --------------------------------------------------------
    # TITLE FEATURES FROM CURRENT BEST PIPELINE
    # --------------------------------------------------------

    df["title_char_length"] = (
        df["title"].str.len()
    )

    df["title_word_count"] = (
        df["title"]
        .str.split()
        .str.len()
        .fillna(0)
    )

    df["title_digit_count"] = (
        df["title"]
        .str.count(r"\d")
    )

    df["title_uppercase_count"] = (
        df["title"]
        .apply(
            lambda text:
            sum(
                char.isupper()
                for char in text
            )
        )
    )

    def first_number(text):

        match = re.search(
            r"\d+(?:\.\d+)?",
            str(text),
        )

        return (
            float(match.group())
            if match
            else 0.0
        )

    df["title_first_number"] = (
        df["title"]
        .apply(first_number)
    )

    lower_title = (
        df["title"]
        .str.lower()
    )

    patterns = {
        "has_gb":
            r"\b\d+(?:\.\d+)?\s*gb\b",

        "has_tb":
            r"\b\d+(?:\.\d+)?\s*tb\b",

        "has_ram":
            r"\b(?:ram|memory)\b",

        "has_inch":
            r'\b\d+(?:\.\d+)?\s*(?:inch|inches|")',

        "has_cm":
            r"\b\d+(?:\.\d+)?\s*cm\b",

        "has_kg":
            r"\b\d+(?:\.\d+)?\s*kg\b",

        "has_gram":
            r"\b\d+(?:\.\d+)?\s*(?:g|gram|grams)\b",

        "has_watt":
            r"\b\d+(?:\.\d+)?\s*(?:w|watt|watts)\b",

        "has_volt":
            r"\b\d+(?:\.\d+)?\s*(?:v|volt|volts)\b",

        "has_pack":
            r"\b(?:pack|set|pair|bundle)\b",

        "has_multipack_number":
            r"\b\d+\s*[- ]?(?:pack|piece|pcs|count|ct)\b",

        "has_pro":
            r"\bpro\b",

        "has_max":
            r"\bmax\b",

        "has_premium":
            r"\bpremium\b",

        "has_professional":
            r"\bprofessional\b",

        "has_wireless":
            r"\bwireless\b",

        "has_smart":
            r"\bsmart\b",
    }

    for feature, pattern in patterns.items():

        df[feature] = (
            lower_title
            .str.contains(
                pattern,
                regex=True,
            )
            .astype(int)
        )

    # --------------------------------------------------------
    # EXTERNAL FEATURES
    # --------------------------------------------------------

    for column in [
        "brand",
        "color",
        "material",
    ]:

        df[column] = (
            df[column]
            .fillna("Unknown")
            .astype(str)
            .str.strip()
        )

        df.loc[
            df[column] == "",
            column,
        ] = "Unknown"

    df["has_abo_brand"] = (
        df["brand"]
        .ne("Unknown")
        .astype(int)
    )

    df["has_abo_color"] = (
        df["color"]
        .ne("Unknown")
        .astype(int)
    )

    df["has_abo_material"] = (
        df["material"]
        .ne("Unknown")
        .astype(int)
    )

    df["mave_attribute_count"] = (
        pd.to_numeric(
            df["mave_attribute_count"],
            errors="coerce",
        )
        .fillna(0)
    )

    df["mave_text"] = (
        df["mave_attributes_json"]
        .apply(
            flatten_mave_attributes
        )
    )

    df["has_mave"] = (
        df["mave_text"]
        .str.len()
        .gt(0)
        .astype(int)
    )

    df["mave_text_length"] = (
        df["mave_text"]
        .str.len()
    )

    # --------------------------------------------------------
    # COMBINED TEXT
    # --------------------------------------------------------

    df["combined_text"] = (
        df["title"]
        + " brand "
        + df["brand"]
        + " material "
        + df["material"]
        + " color "
        + df["color"]
        + " attributes "
        + df["mave_text"]
    )

    return df


train_df = create_features(
    train_df
)

validation_df = create_features(
    validation_df
)

test_df = create_features(
    test_df
)


# ============================================================
# FEATURE CONFIGURATION
# ============================================================

NUMERIC_FEATURES = [
    "stars",
    "reviews_log1p",
    "bought_log1p",
    "isBestSeller",
    "cluster_id",

    "title_char_length",
    "title_word_count",
    "title_digit_count",
    "title_uppercase_count",
    "title_first_number",

    "has_gb",
    "has_tb",
    "has_ram",
    "has_inch",
    "has_cm",
    "has_kg",
    "has_gram",
    "has_watt",
    "has_volt",
    "has_pack",
    "has_multipack_number",

    "has_pro",
    "has_max",
    "has_premium",
    "has_professional",
    "has_wireless",
    "has_smart",

    # External coverage signals
    "has_abo_brand",
    "has_abo_color",
    "has_abo_material",
    "mave_attribute_count",
    "has_mave",
    "mave_text_length",
]


CATEGORICAL_FEATURES = [
    "category_name",
    "brand",
    "material",
    "color",
]


# ============================================================
# PREPROCESS STRUCTURED FEATURES
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES,
        ),

        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=3,
            ),
            CATEGORICAL_FEATURES,
        ),
    ]
)


X_train_structured = (
    preprocessor
    .fit_transform(train_df)
)

X_validation_structured = (
    preprocessor
    .transform(validation_df)
)

X_test_structured = (
    preprocessor
    .transform(test_df)
)


print()
print("=" * 90)
print("STRUCTURED FEATURES")
print("=" * 90)

print(
    "Train:",
    X_train_structured.shape
)


# ============================================================
# TF-IDF ON TITLE + EXTERNAL ATTRIBUTES
# ============================================================

tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",

    ngram_range=(1, 2),

    min_df=3,
    max_df=0.98,

    max_features=10000,

    sublinear_tf=True,

    dtype=np.float32,
)


X_train_text = (
    tfidf.fit_transform(
        train_df["combined_text"]
    )
)

X_validation_text = (
    tfidf.transform(
        validation_df["combined_text"]
    )
)

X_test_text = (
    tfidf.transform(
        test_df["combined_text"]
    )
)


print(
    "TF-IDF:",
    X_train_text.shape
)


# ============================================================
# FINAL FEATURE MATRICES
# ============================================================

X_train = hstack(
    [
        csr_matrix(
            X_train_structured
        ),
        X_train_text,
    ],
    format="csr",
)

X_validation = hstack(
    [
        csr_matrix(
            X_validation_structured
        ),
        X_validation_text,
    ],
    format="csr",
)

X_test = hstack(
    [
        csr_matrix(
            X_test_structured
        ),
        X_test_text,
    ],
    format="csr",
)


print(
    "Final:",
    X_train.shape
)


# ============================================================
# TARGET
# ============================================================

y_train = (
    train_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_validation = (
    validation_df["price"]
    .astype(np.float32)
    .to_numpy()
)

y_test = (
    test_df["price"]
    .astype(np.float32)
    .to_numpy()
)


y_train_log = np.log1p(
    y_train
)

y_validation_log = np.log1p(
    y_validation
)


# ============================================================
# TRAIN V4
# ============================================================

print()
print("=" * 90)
print(
    "TRAINING V4 — ABO/MAVE ENRICHED LOG-LIGHTGBM"
)
print("=" * 90)


model = lgb.LGBMRegressor(
    objective="regression",

    n_estimators=5000,
    learning_rate=0.02,

    num_leaves=63,
    max_depth=-1,

    min_child_samples=20,

    subsample=0.85,
    colsample_bytree=0.85,

    reg_alpha=0.05,
    reg_lambda=1.0,

    random_state=RANDOM_STATE,

    n_jobs=-1,
    verbosity=-1,
)


model.fit(
    X_train,
    y_train_log,

    eval_set=[
        (
            X_validation,
            y_validation_log,
        )
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=200,
            verbose=False,
        )
    ],
)


# ============================================================
# PREDICTIONS
# ============================================================

validation_prediction = np.expm1(
    model.predict(
        X_validation
    )
)

test_prediction = np.expm1(
    model.predict(
        X_test
    )
)

validation_prediction = np.clip(
    validation_prediction,
    0,
    None,
)

test_prediction = np.clip(
    test_prediction,
    0,
    None,
)


# ============================================================
# METRICS
# ============================================================

def calculate_metrics(
    y_true,
    prediction,
):

    return {
        "mae":
            mean_absolute_error(
                y_true,
                prediction,
            ),

        "rmse":
            np.sqrt(
                mean_squared_error(
                    y_true,
                    prediction,
                )
            ),

        "median_ae":
            median_absolute_error(
                y_true,
                prediction,
            ),

        "r2":
            r2_score(
                y_true,
                prediction,
            ),
    }


validation_metrics = (
    calculate_metrics(
        y_validation,
        validation_prediction,
    )
)

test_metrics = (
    calculate_metrics(
        y_test,
        test_prediction,
    )
)


print()
print("VALIDATION")
print("-" * 60)

for key, value in validation_metrics.items():
    print(
        f"{key:12s}: {value:.4f}"
    )


print()
print("TEST")
print("-" * 60)

for key, value in test_metrics.items():
    print(
        f"{key:12s}: {value:.4f}"
    )


# ============================================================
# COMPARISON
# ============================================================

comparison_df = pd.DataFrame(
    [
        {
            "model":
                "V2 Current Champion",

            "validation_mae":
                21.8504,

            "validation_rmse":
                85.0096,

            "validation_median_ae":
                8.0066,

            "validation_r2":
                0.4190,
        },

        {
            "model":
                "V4 ABO/MAVE",

            "validation_mae":
                validation_metrics["mae"],

            "validation_rmse":
                validation_metrics["rmse"],

            "validation_median_ae":
                validation_metrics[
                    "median_ae"
                ],

            "validation_r2":
                validation_metrics["r2"],
        },
    ]
)


print()
print("=" * 90)
print("V2 vs V4")
print("=" * 90)

display(
    comparison_df
)


# ============================================================
# SAVE
# ============================================================

joblib.dump(
    model,
    MODEL_ROOT
    / "v4_external_attributes_lightgbm.joblib",
)

joblib.dump(
    preprocessor,
    MODEL_ROOT
    / "v4_preprocessor.joblib",
)

joblib.dump(
    tfidf,
    MODEL_ROOT
    / "v4_tfidf.joblib",
)

comparison_df.to_csv(
    REPORT_ROOT
    / "v2_vs_v4.csv",
    index=False,
)

print()
print(
    "✅ V4 experiment completed."
)

V4 EXTERNAL ATTRIBUTE ENRICHMENT
Linked dataset:
/Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/linked/multimodal_products_direct.parquet

Model data:
Train: (13984, 26)
Validation: (2997, 26)
Test: (2997, 26)

Linked columns available:
  asin
  title_amazon
  imgUrl
  productURL
  price
  listPrice
  category_id
  category_name
  stars
  reviews
  isBestSeller
  has_valid_price
  price_log1p
  normalized_id
  item_id
  title_abo
  brand
  color
  material
  style
  primary_product_type
  primary_node
  main_image_id
  all_image_ids
  combined_text
  country
  domain_name
  mave_attribute_count
  mave_attribute_value_count
  mave_attributes_json
  mave_attribute_values_text
  has_abo_match
  has_mave_match
  has_three_way_match
  main_image_available
  link_method
  absolute_path
  relative_path
  file_name
  file_size_bytes
  has_local_image
  is_multimodal_ready

External columns selected:
['asin', 'brand', 'color', 'material', '

,split,field,available_rows,total_rows,coverage_percent
0,train,brand,14,13984,0.100114
1,train,color,10,13984,0.071510
2,train,material,7,13984,0.050057
3,train,mave_attributes_json,136,13984,0.972540
4,validation,brand,3,2997,0.100100
5,validation,color,0,2997,0.000000
6,validation,material,1,2997,0.033367
7,validation,mave_attributes_json,23,2997,0.767434
8,test,brand,3,2997,0.100100
9,test,color,2,2997,0.066733



STRUCTURED FEATURES
Train: (13984, 274)
TF-IDF: (13984, 10000)
Final: (13984, 10274)

TRAINING V4 — ABO/MAVE ENRICHED LOG-LIGHTGBM


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



VALIDATION
------------------------------------------------------------
mae         : 21.9318
rmse        : 85.4384
median_ae   : 7.9320
r2          : 0.4131

TEST
------------------------------------------------------------
mae         : 21.8033
rmse        : 65.5678
median_ae   : 7.9256
r2          : 0.5215

V2 vs V4


,model,validation_mae,validation_rmse,validation_median_ae,validation_r2
0,V2 Current Champion,21.850400,85.009600,8.006600,0.419000
1,V4 ABO/MAVE,21.931813,85.438438,7.931972,0.413119



✅ V4 experiment completed.
